In [24]:
import subprocess
import sys
from pathlib import Path
import time

# ---------- Configuration ----------

LOCUSTFILE = "locustfile.py"
HOST = "http://127.0.0.1:8084"

SSH_TARGET = "osboxes@127.0.0.1"
SSH_PORT = "2222"

# LOADS = [1000, 1500, 2000, 2500]
LOADS = [100, 200, 300, 400]
TOTAL_TIME = 15  # seconds
APPLICATIONS = ["cpu", "memory"]
# APPLICATIONS = ['io']

C = "C4"

RESULTS_DIR = Path(f"results/{C}")


def run_test(scenario, application, users):
    test_name = f"{scenario}_{application}_{users}users"
    csv_prefix = RESULTS_DIR / test_name

    metrics_path = RESULTS_DIR / f"{test_name}_vmstat.txt"
    log_path = RESULTS_DIR / f"{test_name}.log"

    # Collect VM metrics for slightly longer than the Locust test.
    metrics_duration = TOTAL_TIME + 3

    metrics_command = [
        "ssh", "-p", SSH_PORT, SSH_TARGET,
        f"timeout {metrics_duration}s vmstat -t -n 1"
    ]

    locust_command = [
        sys.executable, "-m", "locust",
        "-f", LOCUSTFILE,
        "--headless",
        "--host", HOST,
        "--tags", application,
        "--users", str(users),
        "--spawn-rate", str(users),
        "--run-time", f"{TOTAL_TIME}s",
        "--csv", str(csv_prefix),
        "--csv-full-history",
        # "--stop-timeout", "30",
    ]

    print(f"\n{'=' * 60}")
    print(f"Scenario: {scenario}")
    print(f"Application: {application}")
    print(f"Users: {users}")
    print(f"Duration: {TOTAL_TIME}s")
    print(f"{'=' * 60}")

    with open(metrics_path, "w") as metrics_file:
        metrics_process = subprocess.Popen(
            metrics_command,
            stdout=metrics_file,
            stderr=subprocess.STDOUT,
            text=True,
        )

        # Give SSH/vmstat a moment to start before generating load.
        import time
        time.sleep(1)

        with open(log_path, "w") as log:
            result = subprocess.run(
                locust_command,
                stdout=log,
                stderr=subprocess.STDOUT,
                text=True,
            )

        # Wait for metric collection to finish.
        try:
            metrics_process.wait(timeout=TOTAL_TIME + 10)
        except subprocess.TimeoutExpired:
            metrics_process.terminate()
            metrics_process.wait()

    if result.returncode != 0:
        print(f"ERROR: {test_name} failed. Check {log_path}")
        return False

    print(f"Completed: {test_name}")
    print(f"VM metrics: {metrics_path}")
    return True


def main():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    scenario = C

    print(f"\nTarget: {HOST}")
    print(f"Scenario: {scenario}")
    print(f"Planned tests: {len(APPLICATIONS) * len(LOADS)}")

    failures = []

    for application in APPLICATIONS:
        print("Sleeping for 15 secs")
        time.sleep(15)
        for users in LOADS:
            if not run_test(scenario, application, users):
                failures.append(
                    f"{scenario}_{application}_{users}users"
                )

            # print("sleeping for 15 seconds...")
        

    print("\nAll tests processed.")

    if failures:
        print("Failed tests:")
        for test in failures:
            print(f"  - {test}")
    else:
        print("All 12 tests completed successfully.")

    print(f"Results saved in: {RESULTS_DIR.resolve()}")


if __name__ == "__main__":
    main()


Target: http://127.0.0.1:8084
Scenario: C4
Planned tests: 8
Sleeping for 15 secs

Scenario: C4
Application: cpu
Users: 100
Duration: 15s
Completed: C4_cpu_100users
VM metrics: results/C4/C4_cpu_100users_vmstat.txt

Scenario: C4
Application: cpu
Users: 200
Duration: 15s
Completed: C4_cpu_200users
VM metrics: results/C4/C4_cpu_200users_vmstat.txt

Scenario: C4
Application: cpu
Users: 300
Duration: 15s
Completed: C4_cpu_300users
VM metrics: results/C4/C4_cpu_300users_vmstat.txt

Scenario: C4
Application: cpu
Users: 400
Duration: 15s
Completed: C4_cpu_400users
VM metrics: results/C4/C4_cpu_400users_vmstat.txt
Sleeping for 15 secs

Scenario: C4
Application: memory
Users: 100
Duration: 15s
Completed: C4_memory_100users
VM metrics: results/C4/C4_memory_100users_vmstat.txt

Scenario: C4
Application: memory
Users: 200
Duration: 15s
Completed: C4_memory_200users
VM metrics: results/C4/C4_memory_200users_vmstat.txt

Scenario: C4
Application: memory
Users: 300
Duration: 15s
Completed: C4_memory_3